In [17]:
# --- Cell 1: Import Libraries & Config ---
import feedparser
import newspaper
import csv
import hashlib
import time
import requests
from datetime import datetime, timedelta

# 1. Simulate 'sources' Table (GENERIC LINKS NOW)
SOURCES_CONFIG = [
    # The user inputs these into the DB 'base_url' column:
    {"domain_name": "Blognone", "base_url": "https://www.blognone.com"}, 
    {"domain_name": "TechCrunch", "base_url": "https://techcrunch.com"}, 
    {"domain_name": "The Verge", "base_url": "https://www.theverge.com"},
    {"domain_name": "BBC Tech", "base_url": "https://www.bbc.com"},
]

# 2. Simulate 'tags' Table (Keywords)
KEYWORDS_CONFIG = [
    "AI", "Artificial Intelligence", "Machine Learning", 
    "Data", "Google", "Microsoft", "Meta", "NVIDIA", "Crypto",
]

# 3. Settings
DAYS_LOOKBACK = 14 
MIN_CONTENT_LENGTH = 300
OUTPUT_FILENAME = "scraped_data_from_homepages.csv"
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'

print(f"Configuration Loaded. Targets: {[s['domain_name'] for s in SOURCES_CONFIG]}")


Configuration Loaded. Targets: ['Blognone', 'TechCrunch', 'The Verge', 'BBC Tech']


In [14]:
# --- Cell 2: RSS Discovery Logic ---

def resolve_rss_link(homepage_url):
    """
    Takes a homepage (e.g., 'techcrunch.com') and finds the hidden RSS feed.
    Returns the Feed URL string, or None if failed.
    """
    print(f"   🔎 Resolving Feed for: {homepage_url}...", end="")
    
    # List of common hidden locations for RSS
    possible_suffixes = [
        "",             # Maybe it IS the feed?
        "/feed",        # WordPress standard
        "/rss",         # General standard
        "/rss.xml",     # Old school
        "/atom.xml",    # Blognone/GitHub standard
        "/feed.xml",
        "/index.xml"    # Hugo/Static sites
    ]
    
    headers = {'User-Agent': USER_AGENT}

    for suffix in possible_suffixes:
        target = homepage_url.rstrip("/") + suffix
        try:
            # Fast check (Timeout 3s)
            response = requests.get(target, headers=headers, timeout=3)
            
            if response.status_code == 200:
                # Check if it's actually a feed
                d = feedparser.parse(response.content)
                if len(d.entries) > 0:
                    print(f" FOUND! -> {target}")
                    return target
        except Exception:
            continue # Network error, try next suffix
            
    print(" ❌ No Feed Found.")
    return None


In [ ]:
# --- Cell 3: Helper Logic (with DB field preparation) ---

def generate_hash(url):
    """Generate SHA-256 hash for URL deduplication."""
    return hashlib.sha256(url.encode('utf-8')).hexdigest()

def is_date_valid(published_struct):
    if not published_struct: return True
    pub_date = datetime(*published_struct[:6])
    cutoff = datetime.now() - timedelta(days=DAYS_LOOKBACK)
    return pub_date >= cutoff

def get_matched_keywords(headline, text, keyword_list):
    """Return list of matched keywords (for DB junction table)."""
    combined = (headline + " " + text).lower()
    matches = [kw for kw in keyword_list if kw.lower() in combined]
    return matches if matches else []

def normalize_source_name(source_name):
    """Map source domain_name to standard format for DB lookup."""
    # This is a helper for Phase 2 when we query DB for source_id
    return source_name.strip()


In [ ]:
# --- Cell 4: Run Pipeline (with DB Fields) ---

def run_test_pipeline():
    all_data = []
    
    for source in SOURCES_CONFIG:
        print(f"\n--- Processing Source: {source['domain_name']} ---")
        
        # 1. Resolve the Generic Link to an RSS Link
        feed_url = resolve_rss_link(source['base_url'])
        
        if not feed_url:
            print("   [Skip] Could not find a valid RSS feed for this source.")
            continue
            
        # 2. Parse the Feed
        feed = feedparser.parse(feed_url)
        print(f"   📡 Scanning {len(feed.entries)} entries...")
        
        for entry in feed.entries:
            try:
                # Data Mapping
                headline = entry.title
                article_url = entry.link
                
                # Date Filter
                if hasattr(entry, 'published_parsed') and not is_date_valid(entry.published_parsed):
                    continue

                # Content Extraction (Newspaper3k)
                # Note: We wrap this in try/except to prevent one bad link stopping the loop
                art = newspaper.Article(article_url)
                art.download()
                art.parse()
                
                if not art.text or len(art.text) < MIN_CONTENT_LENGTH:
                    continue
                    
                # Keyword Filter & Matching
                matched_keywords = get_matched_keywords(headline, art.text, KEYWORDS_CONFIG)
                if not matched_keywords:
                    continue
                
                # === PHASE 2: DB-Ready Fields ===
                url_hash = generate_hash(article_url)
                published_str = entry.published if hasattr(entry, 'published') else datetime.now().isoformat()
                
                # Success: Build Row (includes all DB fields)
                row = {
                    # === For CSV validation ===
                    "source": source['domain_name'],
                    "headline": headline,
                    "author": art.authors[0] if art.authors else "Unknown",
                    "url": article_url,
                    "published": published_str,
                    "keywords": ", ".join(matched_keywords),
                    "content_snippet": art.text[:100].replace('\n', ' ') + "...",
                    
                    # === DB-Ready Fields (for Phase 2) ===
                    "url_hash": url_hash,
                    "full_content": art.text,  # Full article text for DB
                    "matched_tags": matched_keywords,  # List for junction table
                    "status": "New",  # All articles start as 'New' for AI scoring
                }
                all_data.append(row)
                print(f"   ✅ Keep: {headline[:40]}...")
                
            except Exception as e:
                # print(f"   ❌ Error: {e}")
                continue

    # 3. Save to CSV (with all fields for reference)
    if all_data:
        keys = all_data[0].keys()
        with open(OUTPUT_FILENAME, 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=keys)
            w.writeheader()
            w.writerows(all_data)
        print(f"\n✨ DONE. Saved {len(all_data)} articles to {OUTPUT_FILENAME}")
        print(f"\n📊 CSV Fields Ready for DB:")
        print(f"   ✅ url_hash — For deduplication")
        print(f"   ✅ full_content — For article_content table")
        print(f"   ✅ matched_tags — For article_tag_map junction table")
        print(f"   ✅ status — Initial status 'New' for AI scoring")
    else:
        print("\n⚠️ No relevant articles found.")

run_test_pipeline()



--- Processing Source: Blognone ---
   🔎 Resolving Feed for: https://www.blognone.com... FOUND! -> https://www.blognone.com/atom.xml
   📡 Scanning 10 entries...
 FOUND! -> https://www.blognone.com/atom.xml
   📡 Scanning 10 entries...

--- Processing Source: TechCrunch ---
   🔎 Resolving Feed for: https://techcrunch.com...
--- Processing Source: TechCrunch ---
   🔎 Resolving Feed for: https://techcrunch.com... FOUND! -> https://techcrunch.com/feed
 FOUND! -> https://techcrunch.com/feed
   📡 Scanning 20 entries...
   📡 Scanning 20 entries...
   ✅ Keep: Google details security measures for Chr...
   ✅ Keep: Google details security measures for Chr...
   ✅ Keep: You can buy your Instacart groceries wit...
   ✅ Keep: You can buy your Instacart groceries wit...
   ✅ Keep: TikTok adds a space for organizing conte...
   ✅ Keep: TikTok adds a space for organizing conte...
   ✅ Keep: Petco’s security lapse affected customer...
   ✅ Keep: Petco’s security lapse affected customer...
   ✅ Keep: He